# Sample Analysis

---

## Setup

### Load modules

In [ ]:
# ── Standard libraries ──
import os, sys, copy, random, multiprocessing
import numpy as np
import pandas as pd
import yaml
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Mosaic ──
sys.path = [p for p in sys.path if p]
import missionbio.mosaic as ms

# ── Plotly ──
import plotly.graph_objects as go
import plotly.offline as pyo
pyo.init_notebook_mode()

# ── Interactive tables ──
from itables import init_notebook_mode
import itables.options as opt
init_notebook_mode(all_interactive=True)
opt.lengthMenu = [5, 10, 20, 50, 100, 200, 500]
opt.maxBytes = 0
opt.maxColumns = 0

# ── CPU cores ──
num_cores = max(1, multiprocessing.cpu_count() - 1)

# ── NGT colour scheme ──
wt_col = ms.COLORS[10]
het_col = ms.COLORS[11]
hom_col = ms.COLORS[12]
ms.Config.Colorscale.Dna.NGT = [
    (0 / 4, wt_col), (1 / 4, wt_col),
    (1 / 4, het_col), (2 / 4, het_col),
    (2 / 4, hom_col), (3 / 4, hom_col),
    (3 / 4, 'grey'),  (4 / 4, 'grey'),
]

%matplotlib inline

### Load custom functions

In [ ]:
# Project root
wd = os.path.dirname(os.path.dirname(os.getcwd()))

sys.path.insert(0, os.path.join(wd, "scripts"))
sys.path.insert(0, os.path.join(wd, "scripts", "pipelines"))
import DNA_Utils, PRO_Utils, CNV_Utils, LOH_Pipeline

import importlib
for pkg in [DNA_Utils, PRO_Utils, CNV_Utils, LOH_Pipeline]:
    for name, mod in sorted(sys.modules.items()):
        if name.startswith(pkg.__name__):
            importlib.reload(mod)

# EspressoPro for cell type annotation
try:
    import espressopro as ep
    print("✓ EspressoPro loaded")
except ImportError:
    ep = None
    print("⚠ EspressoPro not available — cell type annotation will be skipped")

verbose=True

print("✓ Custom modules loaded")

### Reproducibility

In [ ]:
%env PYTHONHASHSEED=42
random.seed(42)
np.random.seed(42)

## Configuration

All experiment-level settings from `experiment_config.yml`.
Patient-specific variables set below.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SET EXPERIMENT AND PATIENT — everything else adapts
# ═══════════════════════════════════════════════════════════════════

EXPERIMENT = "MDS001"
PATIENT_ID = "Donor_A"

SKIP_CHECK = True

In [ ]:
# ── Load experiment config ──
config_path = os.path.join(wd, "metadata", EXPERIMENT, "experiment_config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

IS_MULTIPLEXED = config.get("multiplexed", False)
GENOME_BUILD   = config.get("genome_build", "hg19")

# ── Whitelist ──
raw_wl = config.get("whitelist", "")
whitelist = list(dict.fromkeys(raw_wl.split(","))) if raw_wl else []

# ── Germline whitelist ──
germline_wl_path = os.path.join(
    wd,
    config.get(
        "germline_whitelist",
        "resources/common_mutations/Panel_6046_common_HET_variants.txt"
    )
)

with open(germline_wl_path) as f:
    germline_whitelist = [
        line.strip()
        for line in f
        if line.strip() and not line.startswith("#")
    ]

germline_whitelist = list(dict.fromkeys(germline_whitelist))

print(f"Experiment:          {EXPERIMENT}")
print(f"Patient:             {PATIENT_ID}")
print(f"Whitelist:           {len(whitelist)} variants")
print(f"Germline whitelist:  {len(germline_whitelist)} variants")

In [ ]:
# ── Find protein-filtered H5 ──
if IS_MULTIPLEXED:
    h5_dir = os.path.join(wd, "processed_data", "h5", EXPERIMENT, "demultiplexed", PATIENT_ID)
else:
    h5_dir = os.path.join(wd, "processed_data", "h5", EXPERIMENT, PATIENT_ID)

# Prefer protein_filtered, fall back to protein_unfiltered
h5_candidates = sorted([f for f in os.listdir(h5_dir) if f.endswith(".h5")])
h5_filtered = [f for f in h5_candidates if "protein_filtered" in f.lower()]
h5_file = h5_filtered[0] if h5_filtered else h5_candidates[0]

H5_PATH = os.path.join(h5_dir, h5_file)
print(f"H5 file: {H5_PATH}")

# ── Output directories ──
results_base = os.path.join(wd, "results", "per_sample", EXPERIMENT, PATIENT_ID)
pro_outdir = os.path.join(results_base, "protein")
dna_outdir = os.path.join(results_base, "dna_cnv")
for d in [pro_outdir, dna_outdir]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Use dataset size from H5 for dynamic min_mut_prct_cells
sample_raw_size = ms.load(H5_PATH, filter_variants=True, filter_cells=False).dna.shape[0]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

##############################################################################
# Load sample
##############################################################################

sample = ms.load(
    H5_PATH,
    filter_variants=True,
    filter_cells=False,
    variant_filter_config={
        "min_dp": 10,
        "min_gq": 20,
        "vaf_ref": 5,
        "vaf_hom": 95,
        "vaf_het": 30,
        "min_prct_cells": 50,
        "min_mut_prct_cells": max(5, min(20, round(0.01 * sample_raw_size))) * 100 / sample_raw_size,
        "iterations": 10,
        "n_cores": num_cores,
    },
    whitelist=whitelist + germline_whitelist,
)

##############################################################################
# Extract DNA genotype layers
##############################################################################

dna = sample.dna

ngt = dna.get_attribute("NGT")
af = dna.get_attribute("AF")

if not isinstance(ngt, pd.DataFrame):
    cell_ids = dna.row_attrs["barcode"]
    variant_ids = dna.col_attrs["id"]
    ngt = pd.DataFrame(ngt, index=cell_ids, columns=variant_ids)

if not isinstance(af, pd.DataFrame):
    cell_ids = dna.row_attrs["barcode"]
    variant_ids = dna.col_attrs["id"]
    af = pd.DataFrame(af, index=cell_ids, columns=variant_ids)

##############################################################################
# Restrict downstream summaries and plots to whitelist only
##############################################################################

whitelist_only = list(dict.fromkeys(whitelist))
whitelist_only = [v for v in whitelist_only if v in ngt.columns]

missing_whitelist = sorted(set(whitelist) - set(whitelist_only))

if len(whitelist_only) == 0:
    raise ValueError("None of the whitelist variants were found in loaded NGT columns.")

if len(missing_whitelist) > 0:
    print(f"WARNING: {len(missing_whitelist)} whitelist variants were not found in loaded data:")
    print(missing_whitelist)

ngt_wl = ngt.loc[:, whitelist_only]
af_wl = af.loc[:, whitelist_only]

##############################################################################
# Compute per-whitelist-variant genotype and mutation summaries
##############################################################################

genotyped_mask = ngt_wl.isin([0, 1, 2])
mutant_mask = ngt_wl.isin([1, 2])

n_total_cells = ngt_wl.shape[0]
n_genotyped = genotyped_mask.sum(axis=0)
n_mutant = mutant_mask.sum(axis=0)

variant_summary = pd.DataFrame({
    "variant": whitelist_only,
    "n_total_cells": n_total_cells,
    "n_cells_genotyped": n_genotyped.values,
    "pct_cells_genotyped": 100 * n_genotyped.values / n_total_cells,
    "n_mutant_cells": n_mutant.values,
    "pct_mutant_all_cells": 100 * n_mutant.values / n_total_cells,
    "pct_mutant_genotyped_cells": 100 * n_mutant.values / n_genotyped.replace(0, np.nan).values,
    "median_AF_genotyped_cells": af_wl.where(genotyped_mask).median(axis=0).values,
    "median_AF_mutant_cells": af_wl.where(mutant_mask).median(axis=0).values
})

variant_summary["variant"] = pd.Categorical(
    variant_summary["variant"],
    categories=whitelist_only,
    ordered=True
)

variant_summary = variant_summary.sort_values("variant")

In [ ]:
whitelist_only = list(dict.fromkeys(whitelist))
whitelist_only = [v for v in whitelist_only if v in variant_summary["variant"].values]

plot_df = variant_summary[
    variant_summary["variant"].isin(whitelist_only)
].copy()

plot_df["variant"] = pd.Categorical(
    plot_df["variant"],
    categories=whitelist_only,
    ordered=True
)

plot_df = plot_df.sort_values("variant")

plot_long = plot_df.melt(
    id_vars="variant",
    value_vars=[
        "pct_cells_genotyped",
        "pct_mutant_genotyped_cells"
    ],
    var_name="metric",
    value_name="percent"
)

plot_long["metric"] = plot_long["metric"].map({
    "pct_cells_genotyped": "% cells genotyped",
    "pct_mutant_genotyped_cells": "% mutant cells"
})

plt.figure(figsize=(max(10, 0.6 * len(whitelist_only)), 6))

x = np.arange(len(whitelist_only))
width = 0.4

genotyped_vals = plot_df["pct_cells_genotyped"].values
mutant_vals = plot_df["pct_mutant_genotyped_cells"].values

plt.bar(
    x - width / 2,
    genotyped_vals,
    width,
    label="% cells genotyped"
)

plt.bar(
    x + width / 2,
    mutant_vals,
    width,
    label="% mutant cells"
)

plt.xticks(x, plot_df["variant"], rotation=90)
plt.ylabel("Percent")
plt.xlabel("Whitelist variant")
plt.title("% genotyped and % mutant cells per whitelist variant")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.savefig("whitelist_pct_genotyped_pct_mutant.pdf", bbox_inches="tight")
plt.show()

##############################################################################
# Write plotted table
##############################################################################

plot_table = plot_df[
    [
        "variant",
        "n_total_cells",
        "n_cells_genotyped",
        "pct_cells_genotyped",
        "n_mutant_cells",
        "pct_mutant_genotyped_cells",
        "pct_mutant_all_cells"
    ]
].copy()

plot_table = plot_table.rename(columns={
    "variant": "variant",
    "n_total_cells": "n_total_cells",
    "n_cells_genotyped": "n_cells_genotyped",
    "pct_cells_genotyped": "pct_cells_genotyped",
    "n_mutant_cells": "n_mutant_cells",
    "pct_mutant_genotyped_cells": "pct_mutant_cells_among_genotyped",
    "pct_mutant_all_cells": "pct_mutant_cells_among_all"
})

plot_table.to_csv(
    "whitelist_pct_genotyped_pct_mutant_table.csv",
    index=False
)

display(plot_table)

In [ ]:
whitelist = ['chr17:7577535:C/G']

### Resolve paths and load data

In [ ]:
Sample = ms.load(
    H5_PATH,
    filter_variants=True,
    filter_cells=False,
    variant_filter_config={
        "min_dp": 10,
        "min_gq": 20,
        "vaf_ref": 5,
        "vaf_hom": 95,
        "vaf_het": 30,
        "min_prct_cells": 50,
        "min_mut_prct_cells": max(5, min(20, round(0.01 * sample_raw_size))) * 100 / sample_raw_size,
        "iterations": 10,
        "n_cores": num_cores,
    },
    whitelist=whitelist+germline_whitelist,
)
Sample_backup = copy.copy(Sample)
print(f"Loaded: {Sample.dna.shape[0]} cells × {Sample.dna.shape[1]} variants")

### Data overview

In [ ]:
for name, assay in [("DNA", Sample.dna), ("Protein", Sample.protein)]:
    if assay is not None:
        print(f"\n{name}: {assay}")
        print(f"  row_attrs: {list(assay.row_attrs.keys())}")
        print(f"  col_attrs: {list(assay.col_attrs.keys())}")
        print(f"  layers:    {list(assay.layers.keys())}")

## Protein analysis

### Filter non-informative proteins

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — drop isotype controls and non-informative markers
# ═══════════════════════════════════════════════════════════════════
drop_markers = ['IgG1', 'IgG2a', 'IgG2b']
Sample.protein = Sample.protein.drop(drop_markers)
print(f"Protein markers: {Sample.protein.shape[1]} (dropped {drop_markers})")

In [ ]:
# Remove cells with zero protein reads
df_pro = Sample.protein.get_attribute('read_counts', constraint='row+col')
valid_cells = df_pro.loc[df_pro.sum(axis=1) > 0].index
Sample = Sample[valid_cells.tolist()]
print(f"Cells with protein signal: {len(valid_cells)}")

### Normalisation and dimensionality reduction

In [ ]:
random.seed(123); np.random.seed(123)

if ep is not None:
    ep.Normalise_protein_data(Sample, inplace=True, axis=1, flavor="seurat")
    random.seed(123); np.random.seed(123)
    ep.Scale_protein_data(Sample, inplace=True)
else:
    # Manual CLR normalisation fallback
    from scipy.stats import gmean
    rc = Sample.protein.get_attribute('read_counts', constraint='row+col')
    clr = np.log1p(rc.div(rc.apply(gmean, axis=1), axis=0))
    Sample.protein.add_layer('Normalized_reads', clr.values)
    Sample.protein.add_layer('Scaled_reads', ((clr - clr.mean()) / clr.std()).values)

print("✓ Normalised and scaled")

In [ ]:
random.seed(123); np.random.seed(123)
Sample.protein.run_pca(attribute='Scaled_reads', components=8, show_plot=False, random_state=42, svd_solver='randomized')

random.seed(123); np.random.seed(123)
Sample.protein.run_umap(attribute='pca', random_state=42, n_neighbors=50, min_dist=0.1, spread=8, n_components=2)

### Clustering

In [ ]:
random.seed(123); np.random.seed(123)
Sample.protein.cluster(attribute='umap', method='graph-community', k=5, random_state=42)

fig = Sample.protein.scatterplot(attribute='umap', colorby='label')
go.Figure(fig)

In [ ]:
Sample.protein.row_attrs['Clusters'] = copy.copy(Sample.protein.row_attrs['label'])

### Cell type annotation

In [ ]:
if ep is not None:
    Sample = ep.generate_predictions(obj=Sample)
    Sample = ep.annotate_data(obj=Sample)
    fig = Sample.protein.scatterplot(attribute='umap', colorby='Averaged.Detailed.Celltype')
    go.Figure(fig)
else:
    print("⚠ EspressoPro not available — annotate manually")

In [ ]:
if ep is not None:
    fig = Sample.protein.heatmap(attribute='Scaled_reads', splitby='Averaged.Detailed.Celltype')
    go.Figure(fig)

In [ ]:
Sample.protein.row_attrs['label'] = Sample.protein.row_attrs['Averaged.Detailed.Celltype']

### Cluster relabelling

⚠️ **INPUT REQUIRED** — map predicted cell types to final annotations.

In [ ]:
cell_type_to_clusters = {
    'CD8_T': ['CD8_T_Memory', 'CD8_T_Naive', 'MAIT'],
    'CD4_T': ['CD4_T_Naive', 'CD4_T_Memory', 'Treg', 'CD4_CTL'],
    'Other_T': ['GdT'],
    'Progenitor': ['GMP', 'MkP', 'MEP','EoBaMaP','Pre-Pro-B', 'Pro-B', 'LMPP'],
    'HSC_MPP': ['HSC_MPP'],
    'Mast': ['Mast'],
    'cDC': ['cDC1', 'cDC2'],
    'pDC': ['pDC'],
    'Erythroid': ['Erythroblast','ErP'],
    'Pre-Myeloid': ['Myeloid_progenitor','Myeloid_precursor'],
    'CD16_monocyte': ['CD16_Mono'],
    'CD14_monocyte': ['CD14_Mono'],
    'B': ['Pre-B', 'B_Naive', 'B_Memory', 'Immature_B'],
    'NK': ['NK_CD56_bright', 'NK_CD56_dim'],
    'Other': ['Small', 'Mixed'],
    'Plasma': ['Plasma']
}

# Apply mapping
cluster_to_celltype = {}
for ct, clusters in cell_type_to_clusters.items():
    for c in clusters:
        cluster_to_celltype[str(c)] = ct

current_labels = Sample.protein.get_labels()
new_labels = np.array([cluster_to_celltype.get(str(l), l) for l in current_labels], dtype=object)
Sample.protein.row_attrs['Final_annotation'] = new_labels
Sample.protein.row_attrs['label'] = new_labels

In [ ]:
if ep is not None:
    Sample = ep.mark_mixed_clusters(
    Sample,
    label_in="Final_annotation",
    label_out="Final_annotation",
    cluster_col="Clusters",
    min_frequency_threshold=0.35,
)

### Protein visualisation

In [ ]:
resolution = "Final_annotation"
Sample.protein.row_attrs['label'] = Sample.protein.row_attrs[resolution]

# Copy protein UMAP to DNA assay for overlay plots
Sample.dna.row_attrs['umap'] = Sample.protein.row_attrs['umap']

fig = Sample.protein.scatterplot(attribute='umap', colorby=resolution)
go.Figure(fig)

In [ ]:
fig = Sample.protein.signaturemap('Normalized_reads', splitby=resolution)

In [ ]:
# Protein homogeneity/silhouette analysis
df_pro = Sample.protein.get_attribute('Normalized_reads', constraint='row+col')
cols = list(df_pro.columns.values)
df_pro.loc[:, 'label'] = Sample.protein.row_attrs['Final_annotation']
df_pro = df_pro.loc[:, ['label'] + cols]

per_label_stats, global_h = PRO_Utils.plot_protein_homogeneity_silhouette(
    df_pro,
    label_col="label",
    scale=True
)

In [ ]:
pro_df_anno = Sample.protein.get_attribute('Normalized_reads', constraint='row+col')
cols = list(pro_df_anno.columns.values)
pro_df_anno.loc[:, 'label'] = Sample.protein.get_labels()
pro_df_anno = pro_df_anno.loc[:, ['label'] + cols]

pro_df_anno_raw = Sample.protein.get_attribute('read_counts', constraint='row+col')
cols = list(pro_df_anno_raw.columns.values)
pro_df_anno_raw.loc[:, 'label'] = Sample.protein.get_labels()
pro_df_anno_raw = pro_df_anno_raw.loc[:, ['label'] + cols]

## Variant annotation

In [ ]:
n_variants = len(Sample.dna.col_attrs['id'])
annotation_dfs = []
for i in range(0, n_variants, 1000):
    s = copy.copy(Sample)
    s.dna = Sample.dna[:, i:i+1000]
    annotation_dfs.append(s.dna.get_annotations())

merged_annotation_filtered_detected_variants = pd.concat(annotation_dfs, axis=0)

for col, content in merged_annotation_filtered_detected_variants.items():
    Sample.dna.add_col_attr(col, content.values)

print(f"Annotated {len(merged_annotation_filtered_detected_variants)} variants")

## Germline variant identification

In [ ]:
thresholds = DNA_Utils.get_empirical_cohort_germline_thresholds()

step1 = DNA_Utils.run_empirical_all_variant_germline_export(
    Sample,
    experiment=EXPERIMENT,
    patient_id=PATIENT_ID,
    annotation_df=merged_annotation_filtered_detected_variants,
    known_het_variants=germline_whitelist,
    DNA_Utils=DNA_Utils,
    keep_labels=["CD4_T", "CD8_T", "Other_T", "B", "Plasma"],
    thresholds=thresholds,
    out_dir=(
        "/project/meadlab/shared/02__Projects/02b__Tapestri/"
        "resources/variants_patterns_tapestri"
    ),
    descriptive_min_dp=2,
    placement_min_dp=5,
    verbose=True,
    show_head=True,
)

all_variant_germline_df = step1["all_variant_germline_df"]
Sample_T_subset = step1["Sample_T_subset"]
germline_profile_meta = step1["germline_profile_meta"]

germline = step1["germline"]
clean_germline = step1["clean_germline"]
rescued_germline = step1["rescued_germline"]
ref = step1["ref"]

In [ ]:
step2 = DNA_Utils.add_wt_concordance_and_finalize_germline(
    Sample,
    all_variant_germline_df,
    whitelist,
    DNA_Utils=DNA_Utils,
    label="wt",
    thresholds=thresholds,
    max_ado_score=0.8,
    min_clone_cells=5,
    total_barcodes=len(Sample.dna.row_attrs["barcode"]),
    recipe="calibrated_practical_or_confirmed",
    verbose=True,
)

all_variant_germline_df = step2["all_variant_germline_df"]
Sample_wt = step2["Sample_wt"]
clone_df = step2["clone_df"]

germline = step2["germline"]
clean_germline = step2["clean_germline"]
rescued_germline = step2["rescued_germline"]

In [ ]:
step3 = DNA_Utils.run_step2_barcode_cnv_audit_pipeline(
    step2=step2,
    Sample=Sample,
    all_variant_germline_df=all_variant_germline_df,
    CNV_Utils=CNV_Utils,
    DNA_Utils=DNA_Utils,

    experiment=EXPERIMENT,
    patient_id=PATIENT_ID,
    wd=wd,

    Sample_backup=Sample_backup if "Sample_backup" in globals() else None,

    save_dir=(
        "/project/meadlab/shared/02__Projects/02b__Tapestri/"
        "resources/variants_patterns_tapestri"
    ),
    verbose=True,
)

df_AF_all = step3["df_AF_all"]
df_DP_all = step3["df_DP_all"]
df_ALT_all = step3["df_ALT_all"]
df_REF_all = step3["df_REF_all"]
df_cell_clone_labels = step3["df_cell_clone_labels"]

wilcoxon_results_var_raw = step3["wilcoxon_results_var_raw"]
wilcoxon_results_var = step3["wilcoxon_results_var"]

significant_variant_summary = step3["significant_variant_summary"]
sig_variants_ordered = step3["sig_variants_ordered"]
sig_res = step3["sig_res"]

audit_out = step3["audit_out"]
audit_real = step3["audit_real"]

missed_significant_in_all_variant_germline_df = (
    step3["missed_significant_in_all_variant_germline_df"]
)

real_missed_variants_df = step3["real_missed_variants_df"]

cnv_out = step3["cnv_out"]

all_variant_germline_df = step3["all_variant_germline_df"]
filtered_germline_variants = step3["filtered_germline_variants"]
germline_df_for_plot = step3["germline_df_for_plot"]

In [ ]:
cluster_out = DNA_Utils.run_af_clustering_only(
    df_AF_all=df_AF_all,
    filtered_germline_variants=filtered_germline_variants,
    original_other_vars_for_cluster=globals().get("original_other_vars_for_cluster"),
    other_vars=globals().get("other_vars"),
    patient_id=PATIENT_ID,
    params=DNA_Utils.AF_CLUSTER_ONLY_DEFAULT_PARAMS,
    plot=True,
    show_tables=True,
)

variant_af_cluster_df = cluster_out["variant_af_cluster_df"]
af_cluster_summary_df = cluster_out["af_cluster_summary_df"]
filtered_germline_by_cluster_df = cluster_out["filtered_germline_by_cluster_df"]
filtered_germline_cluster_counts_df = cluster_out["filtered_germline_cluster_counts_df"]
silhouette_df = cluster_out["silhouette_df"]

clustered_af_heatmap_variant_table = cluster_out["clustered_af_heatmap_variant_table"]
af_cluster_color_key_df = cluster_out["af_cluster_color_key_df"]
af_cluster_blocks_df = cluster_out["af_cluster_blocks_df"]

af_mat = cluster_out["af_mat"]
labels = cluster_out["labels"]

In [ ]:
AF_CLUSTER_RENAME_MAP = {
    0: "germline_hom",
    1: "germline_het",
    2: "germline_het",
    3: "germline_het",
    4: "germline_wt",
    5: "noisy",
}

LOH_CLUSTER_IDS = [1, 3]   # set to the high-divergence clusters from the check above

rename_out = DNA_Utils.rename_af_clusters_and_update_filtered_germline(
    variant_af_cluster_df=variant_af_cluster_df,
    af_mat=af_mat,
    df_AF_all=df_AF_all,
    filtered_germline_variants=filtered_germline_variants,
    af_cluster_rename_map=AF_CLUSTER_RENAME_MAP,
    all_variant_germline_df=all_variant_germline_df,
    df=globals().get("df", all_variant_germline_df),
    df_review=globals().get("df_review"),
    labels=labels,
    patient_id=PATIENT_ID,

    # Baseline germline evidence, rather than current pooled AF, determines
    # whether a provisional germline_het assignment may remain usable.
    force_original_filtered_germline=False,
    preserve_baseline_germline_evidence=True,
    require_baseline_germline_evidence=True,
    discarded_poor_qc_renamed_cluster_name="discarded_poor_qc",

    # LOH is tracked independently, but unvalidated markers are never returned
    # as active LOH variants.
    loh_cluster_ids=LOH_CLUSTER_IDS,

    plot=True,
    include_variant_tables=True,
)

# ── Standard outputs ────────────────────────────────────────────────────────
renamed_cluster_variant_df = rename_out["renamed_cluster_variant_df"]
renamed_cluster_summary_df = rename_out["renamed_cluster_summary_df"]
renamed_cluster_tables = rename_out["renamed_cluster_tables"]

filtered_germline_variants_before_renamed_cluster_update = (
    rename_out["filtered_germline_variants_before_renamed_cluster_update"]
)
filtered_germline_variants = rename_out["filtered_germline_variants"]
updated_filtered_germline_df = rename_out["updated_filtered_germline_df"]
excluded_blacklisted_pathogenic_df = rename_out[
    "excluded_blacklisted_pathogenic_df"
]

# ── New evidence-gating QC outputs ──────────────────────────────────────────
discarded_poor_qc_df = rename_out["discarded_poor_qc_df"]

# LOH variants are tracked in parallel. These are baseline-supported markers
# only; discarded poor-QC candidates are excluded automatically.
loh_variants = rename_out["loh_variants"]
loh_variant_df = rename_out["loh_variant_df"]

print(
    f"\nStored {len(loh_variants)} validated LOH variants "
    f"from clusters {rename_out['loh_cluster_ids']}"
)
print(
    f"Discarded {len(discarded_poor_qc_df)} cluster-only germline_het "
    "assignment(s) lacking baseline germline evidence."
)

display(renamed_cluster_summary_df)
display(discarded_poor_qc_df)
display(loh_variant_df)


In [ ]:
col_attrs_df = DNA_Utils.visualize_found_germline_variants(
    Sample_T_subset=Sample_T_subset,
    wd=wd,
    MBio_batch=EXPERIMENT,
    Patient_ID=PATIENT_ID,
    variant_ids=filtered_germline_variants,
    variant_df=germline_df_for_plot,
    output_label="germline_het_seed_plus_cnv",
    save_plots=True,
    show_plots=True,
)

print(f"combined union:    {len(filtered_germline_variants):>4}")
print(f"plotted:           {col_attrs_df.shape[0]}")

In [ ]:
germline_variants_dir = f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/variants/germline_variants"
os.makedirs(germline_variants_dir, exist_ok=True)
germline_variants_provisional_list_path = os.path.join(
    germline_variants_dir, "germline_variants_provisional_list.csv"
)

if os.path.exists(germline_variants_provisional_list_path):
    print(f"Already exists, not overwriting -> {germline_variants_provisional_list_path}")
else:
    pd.Series(sorted(filtered_germline_variants), name="variant_id").to_csv(
        germline_variants_provisional_list_path, index=False
    )
    print(f"Saved {len(filtered_germline_variants)} germline variants -> {germline_variants_provisional_list_path}")

In [ ]:
germline_variants_provisional_list_path = os.path.join(
    wd, "results", "per_sample", EXPERIMENT, PATIENT_ID,
    "dna_cnv", "variants", "germline_variants", "germline_variants_provisional_list.csv"
)
filtered_germline_variants = pd.read_csv(germline_variants_provisional_list_path)["variant_id"].tolist()
print(f"Loaded {len(filtered_germline_variants)} germline variants <- {germline_variants_provisional_list_path}")

## Differential (somatic) variant identification

### Subset to non-T cells for somatic variant calling

In [ ]:
# Build AF matrix with cell type annotations
pro_anno = Sample.protein.row_attrs['Final_annotation']

# Exclude T and NK/B for somatic variant calling
non_lymphoid = ~np.isin(pro_anno, ['CD4_T', 'CD8_T', 'Other_T', 'B', 'Plasma'])
non_lymphoid_barcodes = pro_df_anno.index[non_lymphoid].tolist()

Sample_subset = copy.copy(Sample)
Sample_subset = Sample_subset[non_lymphoid_barcodes]
print(f"Non-lymphoid subset: {Sample_subset.dna.shape[0]} cells")

In [ ]:
merged_annotation_filtered_detected_variants

In [ ]:
Sample_subset.dna.layers['NGT_FILTERED'] = Sample_subset.dna.layers['NGT']
wfv = ms.workflows.VariantSubcloneTable(Sample_subset)
df = DNA_Utils.clones_to_dataframe(wfv, variants=whitelist, max_ado_score=0.8, min_clone_size=(5 / len(Sample.dna.row_attrs['barcode']))*100)
display(df)

In [ ]:
# Remove non-informative clone labels
clones = ['missing', 'small', 'ADO', 'FP']
for c in clones:
    cells = Sample_subset.dna.barcodes({c})
    if len(cells) > 0:
        Sample_subset.dna = Sample_subset.dna.drop(cells)
set(Sample_subset.dna.get_labels())

⚠️ **INPUT REQUIRED** — rename preliminary clones based on variant patterns.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — rename preliminary clones
# ═══════════════════════════════════════════════════════════════════
Sample_subset.dna.rename_labels({
    # '1': 'Clone_A',
    # '2': 'Clone_B',
})

In [ ]:
# Propagate protein labels to DNA subset
dna_bars = np.asarray(Sample_subset.dna.barcodes())
pro_bars = np.asarray(Sample_subset.protein.barcodes())
pro_labels = np.asarray(Sample_subset.protein.row_attrs['Final_annotation'])
bar_to_label = dict(zip(pro_bars, pro_labels))
Sample_subset.protein.row_attrs['label'] = np.array(
    [bar_to_label.get(b, 'Unknown') for b in dna_bars], dtype=object
)

In [ ]:
df_AF = Sample_subset.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample_subset.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — set WT label for somatic variant detection
# For samples WITH WT cells: use the WT clone label
# For samples WITHOUT WT: set to None (uses all cells as reference)
# ═══════════════════════════════════════════════════════════════════
WT_LABEL = "WT"  # e.g. "TP53_Het" or "WT" or None

out = DNA_Utils.find_somatic_variants(
    df_AF, whitelist, filtered_germline_variants, Sample_subset,
    annotation_df=merged_annotation_filtered_detected_variants, 
    wt_label=WT_LABEL,
    always_keep_pathogenic=True,
    pathogenic_bypass_germline_filter=True
)
differential_variants = out[0]

viz = DNA_Utils.visualize_found_differential_variants(
    df=df_AF,
    differential_variants=differential_variants,
    Sample_subset=Sample_subset,
    wd=wd, MBio_batch=EXPERIMENT, Patient_ID=PATIENT_ID,
    Sample=Sample,
    experiment=EXPERIMENT,
    variant_metadata=merged_annotation_filtered_detected_variants,
    resolution="Final_annotation", 
    ncols=4,
)

###########################

from pathlib import Path

somatic_variant_outdir = Path(
    f"/project/meadlab/shared/02__Projects/02b__Tapestri/"
    f"results/per_sample/{EXPERIMENT}/{PATIENT_ID}/"
    "dna_cnv/variants/somatic_variants"
)

somatic_variant_outdir.mkdir(
    parents=True,
    exist_ok=True
)

putative_somatic_variants_path = (
    somatic_variant_outdir /
    "Putative_somatic_variants.csv"
)

# Preserve the exact differential-variant ordering returned by
# DNA_Utils.find_somatic_variants().
differential_variants = list(differential_variants)

# Detect the annotation column that contains the variant IDs.
possible_variant_cols = [
    "variant",
    "Variant",
    "variant_id",
    "Variant_ID",
    "id",
    "ID",
    "name",
    "Name",
]

variant_col = next(
    (
        col for col in possible_variant_cols
        if col in merged_annotation_filtered_detected_variants.columns
    ),
    None
)

if variant_col is None:
    # Often the variant IDs are stored as the dataframe index.
    annotation_export = (
        merged_annotation_filtered_detected_variants
        .copy()
        .reset_index()
    )

    index_variant_col = annotation_export.columns[0]

    matched_variants = [
        variant
        for variant in differential_variants
        if variant in annotation_export[index_variant_col].astype(str).values
    ]

    putative_somatic_variants = (
        annotation_export
        .assign(
            _variant_order=pd.Categorical(
                annotation_export[index_variant_col].astype(str),
                categories=matched_variants,
                ordered=True
            )
        )
        .loc[
            lambda x: x[index_variant_col].astype(str).isin(matched_variants)
        ]
        .sort_values("_variant_order")
        .drop(columns="_variant_order")
    )

else:
    annotation_export = merged_annotation_filtered_detected_variants.copy()

    matched_variants = [
        variant
        for variant in differential_variants
        if variant in annotation_export[variant_col].astype(str).values
    ]

    putative_somatic_variants = (
        annotation_export
        .assign(
            _variant_order=pd.Categorical(
                annotation_export[variant_col].astype(str),
                categories=matched_variants,
                ordered=True
            )
        )
        .loc[
            lambda x: x[variant_col].astype(str).isin(matched_variants)
        ]
        .sort_values("_variant_order")
        .drop(columns="_variant_order")
    )

# Add a column reflecting the order in which variants were identified
putative_somatic_variants.insert(
    0,
    "somatic_variant_rank",
    range(1, len(putative_somatic_variants) + 1)
)

putative_somatic_variants.to_csv(
    putative_somatic_variants_path,
    index=False
)

missing_from_annotation = set(differential_variants) - set(matched_variants)

print(f"Found differential variants: {len(differential_variants)}")
print(f"Exported annotated variants: {len(putative_somatic_variants)}")
print(f"Saved: {putative_somatic_variants_path}")

if missing_from_annotation:
    print("\nWarning: these differential variants were not found in the annotation table:")
    print(sorted(missing_from_annotation))

putative_somatic_variants

###########################

protein_umap_fig = viz["protein_umap_fig"]
variant_grid_fig = viz["variant_grid_fig"]

⚠️ **INPUT REQUIRED** — confirm the differential/somatic variants.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — final somatic variant list
# ═══════════════════════════════════════════════════════════════════
differential_variants = [
    'chr17:7577535:C/G'
]

In [ ]:
# Combine germline + somatic into informative variants
informative_variants = np.unique(sorted(
    np.concatenate((filtered_germline_variants, differential_variants))
))
informative_variants_sorted = sorted(informative_variants, key=DNA_Utils.extract_chr_num)
print(f"Informative variants: {len(informative_variants_sorted)}")

fig, df_filtered = DNA_Utils.visualize_found_informative_variants(
    df=df_AF,
    combined_useful_variants_sorted=informative_variants,
    Sample_subset=Sample_subset,
    wd=wd,
    MBio_batch=EXPERIMENT,
    Patient_ID=PATIENT_ID,
    show_plot=True
)

In [ ]:
# Subset to informative variants only
Sample.dna = Sample.dna[:, informative_variants_sorted]

# Sync protein annotations
PRO_Utils.update_protein_labels_from_dataframe(
    Sample=Sample,
    anno_df=pro_df_anno,
    label_col='label',
    target_attr='label',
    verbose=True
)

## Clonal annotation

In [ ]:
merged_annotation_filtered_detected_variants

In [ ]:
Sample.dna.layers['NGT_FILTERED'] = Sample.dna.layers['NGT']
wfv = ms.workflows.VariantSubcloneTable(Sample)
df = DNA_Utils.clones_to_dataframe(wfv, variants=differential_variants, max_ado_score=0.8, min_clone_size=(5 / len(Sample.dna.row_attrs['barcode']))*100)
display(df)

In [ ]:
clones = ['missing', 'ADO', 'FP', 'small']
for c in clones:
    cells = Sample.dna.barcodes({c})
    if len(cells) > 0:
        Sample.dna = Sample.dna.drop(cells)
set(Sample.dna.get_labels())

In [ ]:
# Somatic variant outlier filtering
df_AF = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

kept_barcodes, all_audits, removed_barcodes_by_variant = DNA_Utils.filter_somatic_variant_outliers(
    Sample=Sample,
    df_AF=df_AF,
    differential_variants=differential_variants,
    clone_variant_calls=df,
    outdir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/variants/somatic_variants",
    dp_threshold=3,
    sd_k=1.25,
)

In [ ]:
# Every cell that failed DP or was an AF outlier at a variant its clone carries
removed_union = set()
for var, idx in removed_barcodes_by_variant.items():
    removed_union |= set(idx.astype(str))

dna_barcodes = np.array(Sample.dna.row_attrs["barcode"]).astype(str)
protein_barcodes = set(np.array(Sample.protein.row_attrs["barcode"]).astype(str))

shared_barcodes = [bc for bc in dna_barcodes if bc not in removed_union and bc in protein_barcodes]

if len(shared_barcodes) == 0:
    raise ValueError("No barcodes remain after removal + DNA/protein intersection.")

n_removed = len(removed_union & set(dna_barcodes))
print(f"Before: {len(dna_barcodes)}  |  removed (DP-fail + SD-outlier): {n_removed}  "
      f"|  after: {len(shared_barcodes)}  ({round(len(shared_barcodes)/len(dna_barcodes)*100,1)}%)")

Sample.dna = Sample.dna[shared_barcodes, :]
Sample.protein = Sample.protein[shared_barcodes, :]
print(f"Sample.dna: {Sample.dna.shape}, Sample.protein: {Sample.protein.shape}")

In [ ]:
figures = DNA_Utils.plot_individual_clone_umaps(
    Sample=Sample,
    clone_column='label',
    outdir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/variants/somatic_variants",
    output_prefix="Clone_UMAP",
    umap_attribute="umap",
    background_color='#adadad',  # Light grey
    clone_size=6,
    background_size=3,
    width=200,
    height=200,
    verbose=True
)

⚠️ **INPUT REQUIRED** — rename clones with biological identifiers.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — clone nomenclature
# ═══════════════════════════════════════════════════════════════════
Sample.dna.rename_labels({
    'TP53_Hom': 'TP53_R249T_Hem',
    'TP53_Het': 'ADO',
})

# Remove residual non-informative clones
clones = ['missing', 'ADO', 'FP', 'small']
for c in clones:
    cells = Sample.dna.barcodes({c})
    if len(cells) > 0:
        Sample.dna = Sample.dna.drop(cells)
set(Sample.dna.get_labels())

Sample = Sample[list(Sample.dna.row_attrs['barcode'])]
Sample.dna.row_attrs['Clone'] = Sample.dna.row_attrs['label']

fig = Sample.dna.heatmap(attribute='AF', override=True, splitby=Sample.dna.row_attrs['label'])
go.Figure(fig)

## CNV analysis

### ALT/REF reads and variant locations

In [ ]:
df_NGT = Sample.dna.get_attribute('NGT', constraint='row+col')
cols = list(df_NGT.columns.values)
df_NGT.loc[:, 'label'] = Sample.dna.get_labels()
df_NGT = df_NGT.loc[:, ['label'] + cols]

df_AF = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

df_dp = Sample.dna.get_attribute('DP', constraint='row+col')
cols_dp = list(df_dp.columns.values)
df_dp.loc[:, 'label'] = Sample.dna.get_labels()
df_dp = df_dp.loc[:, ['label'] + cols_dp]

AF = df_AF.drop(columns=["label"]).fillna(0)
DP = df_dp.drop(columns=["label"]).fillna(0)
df_ALT = (AF / 100 * DP).round().astype(int)
df_REF = (DP - df_ALT).clip(lower=0).astype(int)

In [ ]:
# Cytobands
cytobands = pd.read_table(os.path.join(wd, 'resources', 'references', 'cytobands.tsv'), index_col=[0])

# Group variants by chromosome arm
grouped_by_chr, grouped_by_chr_arm = CNV_Utils.group_informative_variants_by_chr_arm(
    filtered_germline_in_samples=filtered_germline_variants,
    cytobands=cytobands,
    verbose=True
)

# Access results
print(f"\nChromosomes: {list(grouped_by_chr.keys())}")
print(f"Chromosome arms: {list(grouped_by_chr_arm.keys())}")

### LOH pipeline

⚠️ **INPUT REQUIRED** — configure LOH parameters for this sample.

In [ ]:
OUTPUT_DIR = (
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}"
    "/dna_cnv/clonotypes/allele_frequencies"
)

cfg = LOH_Pipeline.build_config(
    cytobands=cytobands,
    informative_variants=informative_variants,
    output_dir=OUTPUT_DIR,
    plot_dropped_clusters=True,
    verbose=True,
    phase1_show_failed_clones=False,
)

print("✓ LOH config built from standard pipeline defaults")

In [ ]:
out = LOH_Pipeline.run(
    Sample=Sample,
    df=df_AF,
    grouped_by_chr_arm=grouped_by_chr_arm,
    chr_arms=None,
    cfg=cfg,

    run_plotting=True,
    informative_variants=informative_variants,
    cytobands=cytobands,
    output_dir=OUTPUT_DIR,
    output_prefix="All_clones_AF",
    include_non_loh=True,
    plot_separate_clones=True,
    split_by_cytobands=False,
    save_figures=True,
    visualise_fn=CNV_Utils.visualise_variants_by_clone,
    print_progress=True,
    post_phase5_subclone_scan_labels="auto",
    post_phase5_subclone_scan_chr_arms=None,
    post_phase5_subclone_scan_min_parent_cells=5,
    post_phase5_subclone_scan_merge_back=True
)

In [ ]:
from LOH_Pipeline.loh_config import LOH_TAG

label_map = {
    '': ''
}
clones_to_drop = {"missing", "ADO", "FP", "small"}

out_labels = (
    out["df_plot_all"]["label"]
    .astype(str)
    .str.strip()
    .replace(label_map)
)
out_labels.index = out_labels.index.astype(str)

barcodes = np.asarray(Sample.dna.row_attrs["barcode"]).astype(str)
clone_series = out_labels.reindex(barcodes)
drop_mask = clone_series.isna() | clone_series.isin(clones_to_drop)
keep_mask = (~drop_mask).to_numpy()
keep_barcodes = barcodes[keep_mask]
keep_clone_values = clone_series.loc[keep_barcodes].to_numpy()

print("Before:", Sample.dna.shape[0])
print("Expected after:", len(keep_clone_values))
print("")
print(pd.Series(keep_clone_values).value_counts(dropna=False))

keep_idx = np.flatnonzero(keep_mask)

n_rows = Sample.dna.shape[0]
stale_keys = [
    k for k, v in Sample.dna.row_attrs.items()
    if hasattr(v, '__len__') and len(v) != n_rows
]
if stale_keys:
    print(f"Dropping stale row_attrs before subset: {stale_keys}")
    for k in stale_keys:
        del Sample.dna.row_attrs[k]

Sample.dna.select_rows(keep_idx)
print("")
print("After dna subset:", Sample.dna.shape[0])
print("")

Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()
Sample.dna.set_labels(keep_clone_values.copy())
Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()

print(pd.Series(Sample.dna.row_attrs["Clone"]).value_counts(dropna=False))
print("")
print(np.unique(Sample.dna.row_attrs["Clone"]))

## Verifying truncating double deletions

In [ ]:
Sample_all_vars = copy.copy(Sample_backup)
Sample_all_vars.cnv.normalize_reads()

desired_plotting_order = list(set(Sample.dna.row_attrs['Clone']))

out_all = CNV_Utils.extract_cnv_data(
    Sample=Sample_all_vars,
    desired_plotting_order=desired_plotting_order,
    wd=wd,
    EXPERIMENT=EXPERIMENT,
    PATIENT_ID=PATIENT_ID,
    diploid_label="WT",
    sample_filter_source=Sample,
    label_source_sample=Sample,
    subset_cnv_to_variant_amplicons=False,
    ploidy_before_sync=False,
    output_tag="_all_vars",
    save_outputs=True,
    verbose=True,
)
df_ploidy_median = out_all["df_ploidy_median"]

In [ ]:
GENOME_BUILD = "hg19"

gtf_path = os.path.join(
    wd,
    "resources",
    "genome",
    GENOME_BUILD,
    f"{GENOME_BUILD}.ncbiRefSeq.gtf",
)

if not os.path.exists(gtf_path):
    raise FileNotFoundError(f"GTF not found: {gtf_path}")

print(f"✓ GTF path: {gtf_path}")

gtf_exons_refseq = CNV_Utils.load_refseq_gtf_exons(gtf_path)

print(f"✓ gtf_exons_refseq loaded: {gtf_exons_refseq.shape}")
display(gtf_exons_refseq.head())

In [ ]:
ploidy_inputs = CNV_Utils.prepare_ploidy_region_plot_inputs(
    sample_all_vars=Sample_all_vars,
    df_ploidy_median=df_ploidy_median,
    annotation_df=merged_annotation_filtered_detected_variants,
    differential_variants=differential_variants,
    desired_plotting_order=desired_plotting_order,
    gtf_exons_refseq=gtf_exons_refseq,
    verbose=True,
)

plot_df = ploidy_inputs["plot_df"]
amp_sel = ploidy_inputs["amp_sel"]
diff = ploidy_inputs["diff"]
gene_to_transcript = ploidy_inputs["gene_to_transcript"]
transcript_regions = ploidy_inputs["transcript_regions"]
genes_of_interest = ploidy_inputs["genes_of_interest"]

###

display(
    diff[
        [
            "query_variant_id",
            "variant_id",
            "matched_variant_id",
            "gene_name_plot",
            "refseq_transcript_id",
            "transcript_id_gtf",
            "CHROM_norm",
            "variant_pos",
            "exon",
            "intron",
            "region_label",
            "mutation_label_exon",
            "exon_match_status",
        ]
    ]
)

display(
    pd.DataFrame(
        [
            {"gene_name": gene, "selected_refseq_transcript": tx}
            for gene, tx in gene_to_transcript.items()
        ]
    )
)

display(
    amp_sel[
        [
            "amplicon",
            "gene_name",
            "CHROM_norm",
            "start_pos",
            "end_pos",
            "exon",
            "intron",
            "region_label",
            "region_rank",
            "exon_source",
            "mutation_labels",
        ]
    ]
)

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def _clean_label(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() in {"", "nan", "none", "na"}:
        return ""

    return x


def _amp_axis_label(row):
    region_label = _clean_label(row.get("region_label", ""))

    if region_label:
        return f"{row['amplicon']} / {region_label}"

    exon = _clean_label(row.get("exon", ""))

    if exon:
        return f"{row['amplicon']} / Exon {exon}"

    intron = _clean_label(row.get("intron", ""))

    if intron:
        return f"{row['amplicon']} / Intron {intron}"

    return str(row["amplicon"])


def plot_gene_ploidy_heatmap(
    gene,
    gene_amp_sel,
    plot_df,
    output_dir,
    cell_size=0.75,
    min_width=8,
    min_height=4.5,
    max_width=None,
    max_height=None,
):
    os.makedirs(output_dir, exist_ok=True)

    gene_amp_sel = (
        gene_amp_sel
        .copy()
        .assign(
            region_rank=lambda d: pd.to_numeric(
                d["region_rank"],
                errors="coerce",
            ).fillna(10**9)
        )
        .sort_values(
            ["region_rank", "start_pos", "end_pos", "amplicon"],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    gene_amp_ids = gene_amp_sel["amplicon"].tolist()
    gene_plot_df = plot_df[gene_amp_ids].copy()

    n_cols = len(gene_amp_ids)
    n_rows = gene_plot_df.shape[0]

    left_right_margin = 2.2
    top_bottom_margin = 2.8

    fig_w = left_right_margin + (n_cols * cell_size)
    fig_h = top_bottom_margin + (n_rows * cell_size)

    fig_w = max(min_width, fig_w)
    fig_h = max(min_height, fig_h)

    if max_width is not None:
        fig_w = min(max_width, fig_w)

    if max_height is not None:
        fig_h = min(max_height, fig_h)

    cmap = plt.cm.get_cmap("RdBu_r").copy()
    cmap.set_under("black")

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        gene_plot_df,
        ax=ax,
        cmap=cmap,
        center=2,
        vmin=0.01,
        vmax=4,
        linewidths=0.35,
        linecolor="white",
        xticklabels=[_amp_axis_label(row) for _, row in gene_amp_sel.iterrows()],
        yticklabels=True,
        square=True,
        cbar_kws={
            "label": "Ploidy",
            "shrink": 0.45,
            "pad": 0.02,
        },
    )

    ax.set_title(f"{gene}", fontsize=14, pad=36)
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=90,
        fontsize=7,
        ha="center",
        va="top",
    )

    ax.set_yticklabels(
        ax.get_yticklabels(),
        fontsize=8,
    )

    for j, row in gene_amp_sel.reset_index(drop=True).iterrows():
        muts = _clean_label(row.get("mutation_labels", ""))

        if not muts:
            continue

        ax.text(
            j + 0.5,
            -0.55,
            muts,
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=8,
            fontweight="bold",
            color="crimson",
            clip_on=False,
        )

        ax.scatter(
            j + 0.5,
            -0.12,
            marker="v",
            s=45,
            color="crimson",
            clip_on=False,
            zorder=10,
        )

    plt.tight_layout(rect=[0, 0.02, 1, 0.94])

    safe_gene = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(gene))

    out_pdf = os.path.join(
        output_dir,
        f"clone_amplicon_ploidy_heatmap_{safe_gene}_selected_refseq_exon_intron_order.pdf",
    )

    out_png = os.path.join(
        output_dir,
        f"clone_amplicon_ploidy_heatmap_{safe_gene}_selected_refseq_exon_intron_order.png",
    )

    plt.savefig(out_pdf, bbox_inches="tight", dpi=150)
    plt.savefig(out_png, bbox_inches="tight", dpi=220)
    plt.show()

    print(f"✓ Saved: {out_pdf}")
    print(f"✓ Saved: {out_png}")

    return {
        "gene": gene,
        "pdf": out_pdf,
        "png": out_png,
        "n_amplicons": n_cols,
        "n_clones": n_rows,
        "fig_width": fig_w,
        "fig_height": fig_h,
    }


plot_outputs = []

for gene in genes_of_interest:
    gene_amp_sel = (
        amp_sel[amp_sel["gene_name"].astype(str) == str(gene)]
        .copy()
        .assign(
            region_rank=lambda d: pd.to_numeric(
                d["region_rank"],
                errors="coerce",
            ).fillna(10**9)
        )
        .sort_values(
            ["region_rank", "start_pos", "end_pos", "amplicon"],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    if gene_amp_sel.empty:
        continue

    out = plot_gene_ploidy_heatmap(
        gene=gene,
        gene_amp_sel=gene_amp_sel,
        plot_df=plot_df,
        output_dir=OUTPUT_DIR,
        cell_size=0.75,
    )

    plot_outputs.append(out)

display(pd.DataFrame(plot_outputs))

⚠️ **INPUT REQUIRED** — rename clones after LOH analysis.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Generate editable label_map template from current clone labels
# ═══════════════════════════════════════════════════════════════════

import re
import pandas as pd
import numpy as np

clones_to_drop = {"missing", "ADO", "FP", "small"}

clone_series_current = (
    pd.Series(Sample.dna.row_attrs["Clone"])
    .astype(str)
    .str.strip()
)

clone_counts = clone_series_current.value_counts(dropna=False)

# Prefer an existing plotting/order object if available.
if "desired_plotting_order" in globals():
    ordered_clones = [
        str(c).strip()
        for c in desired_plotting_order
        if str(c).strip() in set(clone_series_current)
    ]

    # Add any clones not already in desired_plotting_order.
    remaining = [
        c for c in clone_counts.index.astype(str).tolist()
        if c not in ordered_clones
    ]
    ordered_clones += remaining

elif "clone_order" in globals():
    ordered_clones = [
        str(c).strip()
        for c in clone_order
        if str(c).strip() in set(clone_series_current)
    ]

    remaining = [
        c for c in clone_counts.index.astype(str).tolist()
        if c not in ordered_clones
    ]
    ordered_clones += remaining

else:
    # Natural-ish sort: Clone 2 before Clone 10.
    def _natural_key(x):
        x = str(x)
        return [
            int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", x)
        ]

    ordered_clones = sorted(
        clone_counts.index.astype(str).tolist(),
        key=_natural_key,
    )

# Optional: keep drop labels at the bottom.
ordered_clones = [
    c for c in ordered_clones if c not in clones_to_drop
] + [
    c for c in ordered_clones if c in clones_to_drop
]

print("\nCopy/paste this into your renaming cell and fill the values:")
print("label_map = {")
for c in ordered_clones:
    print(f"    {c!r}: '',")
print("}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — post-LOH clone renaming
# ═══════════════════════════════════════════════════════════════════
from LOH_Pipeline.loh_config import LOH_TAG

label_map = {
    'TP53_R249T_Hem LOH-C0 chr7q:q22.1-q36.1 LOH-C0 chr5q:q15-q33.1': 'TP53_R249T_Hem',
    'TP53_R249T_Hem LOH-C0 chr5q:q15-q31.2': 'TP53_R249T_Hem',
    'WT': 'WT',
}
clones_to_drop = {"missing", "ADO", "FP", "small"}

# Current clone labels
clone_series = (
    pd.Series(Sample.dna.row_attrs["Clone"])
    .astype(str)
    .str.strip()
    .replace(label_map)
)

drop_mask = clone_series.isin(clones_to_drop).to_numpy()
keep_mask = ~drop_mask

keep_clone_values = clone_series.loc[keep_mask].to_numpy()

print("Before:", Sample.dna.shape[0])
print("Expected after:", len(keep_clone_values))
print(pd.Series(keep_clone_values).value_counts(dropna=False))

# If no cells need dropping, only rename labels in place
if drop_mask.sum() == 0:
    Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()
    Sample.dna.set_labels(keep_clone_values.copy())
    Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()

else:
    keep_idx = np.flatnonzero(keep_mask)

    # Remove stale row_attrs before subsetting
    n_rows = Sample.dna.shape[0]
    stale_keys = [
        k for k, v in Sample.dna.row_attrs.items()
        if hasattr(v, "__len__") and len(v) != n_rows
    ]

    if stale_keys:
        print(f"Dropping stale row_attrs before subset: {stale_keys}")
        for k in stale_keys:
            del Sample.dna.row_attrs[k]

    # Subset DNA rows in place
    Sample.dna.select_rows(keep_idx)

    print("After dna subset:", Sample.dna.shape[0])

    # Assign renamed labels
    Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()
    Sample.dna.set_labels(keep_clone_values.copy())
    Sample.dna.row_attrs["Clone"] = keep_clone_values.copy()

print(pd.Series(Sample.dna.row_attrs["Clone"]).value_counts(dropna=False))
print(np.unique(Sample.dna.row_attrs["Clone"]))

In [ ]:
final = pd.Series(Sample.dna.row_attrs['Clone']).astype(str).str.strip()
print(f"\n{'='*60}")
print(f"Final clones ({len(final.unique())} labels, {len(final)} cells):")
for lbl in sorted(final.unique(), key=lambda x: -(final == x).sum()):
    print(f"  {lbl:<60s} {(final == lbl).sum():>5d}")
print(f"{'='*60}")

### CNV visualisation

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDIT THIS CELL — desired clone plotting order
# ═══════════════════════════════════════════════════════════════════
desired_plotting_order = [
    'WT',
    'TP53_R249T_Hem'
]

clones = [c for c in desired_plotting_order if c in set(Sample.dna.get_labels())]

#### Visualising all variants

In [ ]:
df_AF = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

CNV_Utils.visualise_variants_by_clone(
    df=df_AF,
    variants_to_plot=informative_variants,
    cytobands=cytobands,
    output_dir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/allele_frequencies",
    output_prefix="Continuous_AF_Single_cell_clones_x_ALL_Informative_Variants",
    label_col='label',
    desired_clone_plotting_order=desired_plotting_order,
    verbose=True,
    plot_separate_clones=True,
    split_by_cytobands=False
)

#### Identifying and visualising extreme variants

In [ ]:
variant_to_amplicon = dict(
    zip(
        np.asarray(Sample.dna.col_attrs["id"]).astype(str),
        np.asarray(Sample.dna.col_attrs["amplicon"]).astype(str),
    )
)

wilcoxon_results_var = CNV_Utils.test_variant_informativeness(
    df_AF=df_AF,
    df_ALT=df_ALT,
    df_REF=df_REF,
    grouped_by_chr_arm=grouped_by_chr_arm,
    output_dir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/allele_frequencies",
    p_threshold=0.01, baf_deviation_threshold=0.3,
    depth_ratio_deletion=0.7, depth_ratio_amplification=1.3,
    require_clone_imbalance=False,
    max_af_iqr=0.35, min_cells=5,
    variant_to_amplicon=variant_to_amplicon,
    germline_variants=list(filtered_germline_variants),
    verbose=True,
)

print("")

variable_af_germline = []
non_variable_af_germline = []

for chr_name, res_df in wilcoxon_results_var.items():
    if not res_df.empty:
        variable_af_germline.extend(res_df[res_df['informative']]['variant'].tolist())
        non_variable_af_germline.extend(res_df[~res_df['informative']]['variant'].tolist())

# Remove duplicates if any
variable_af_germline = list(set(variable_af_germline))
# Remove any variable_germline variants from non_variable_germline
non_variable_af_germline = list(set(non_variable_af_germline) - set(variable_af_germline))

print("Variable AF germline variants (informative):", variable_af_germline)
print("Non-variable AF germline variants:", non_variable_af_germline)

In [ ]:
df_AF = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

CNV_Utils.visualise_variants_by_clone(
    df=df_AF,
    variants_to_plot=variable_af_germline,
    cytobands=cytobands,
    output_dir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/allele_frequencies",
    output_prefix="Continuous_AF_Single_cell_clones_x_ALL_Informative_Variants",
    label_col='label',
    desired_clone_plotting_order=desired_plotting_order,
    verbose=True,
    plot_separate_clones=True,
    split_by_cytobands=False
)

#### Recovering, identifying and visualising lower coverage extreme variants

In [ ]:
res = CNV_Utils.expand_informative_variants(
    H5_PATH,
    Sample,
    cytobands,
    output_dir=(
        f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}"
        "/dna_cnv/clonotypes/allele_frequencies"
    ),
    desired_plotting_order=desired_plotting_order,
    whitelist=whitelist,
    germline_whitelist=germline_whitelist,
    sample_raw_size=sample_raw_size,
    num_cores=num_cores,
    quality_percentile=1,
    adaptive_qc_percentile=1,
    adaptive_median_dp_cap=10.0,
    adaptive_mean_gq_cap=50.0,
    previous_informative_variants=variable_af_germline,
    require_wt_het=True,
    wt_het_band=None,
    wt_het_band_percentiles=(1, 99),
    wt_label="WT",
    blacklist_paths=[
        "/project/meadlab/shared/02__Projects/02__TP53_MDS_Tapestri/Resources/Blacklist/hg19-blacklist.v2.bed",
        "/project/meadlab/shared/02__Projects/02__TP53_MDS_Tapestri/Resources/Blacklist/hg19_blacklist_MergedList_Anshul Kundaje_Stanford.csv",
    ],
    exclude_pathogenic=True,
)
Sample = res["Sample_object"]
print("Kept new variants:", res["new_kept"])
print("Final DNA shape:", Sample.dna.shape)
print("WT-HET band:", res["thresholds"].get("wt_het_band"))
print("QC thresholds:", res["thresholds"])
print(res["additional"][["clone_call_rate", "wt_af", "median_DP", "kept"]].round(3))

In [ ]:
df_NGT = Sample.dna.get_attribute('NGT', constraint='row+col')
cols = list(df_NGT.columns.values)
df_NGT.loc[:, 'label'] = Sample.dna.get_labels()
df_NGT = df_NGT.loc[:, ['label'] + cols]

df_AF = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df_AF.columns.values)
df_AF.loc[:, 'label'] = Sample.dna.get_labels()
df_AF = df_AF.loc[:, ['label'] + cols]

df_dp = Sample.dna.get_attribute('DP', constraint='row+col')
cols_dp = list(df_dp.columns.values)
df_dp.loc[:, 'label'] = Sample.dna.get_labels()
df_dp = df_dp.loc[:, ['label'] + cols_dp]

AF = df_AF.drop(columns=["label"]).fillna(0)
DP = df_dp.drop(columns=["label"]).fillna(0)
df_ALT = (AF / 100 * DP).round().astype(int)
df_REF = (DP - df_ALT).clip(lower=0).astype(int)

grouped_by_chr, grouped_by_chr_arm = CNV_Utils.group_informative_variants_by_chr_arm(
    filtered_germline_in_samples=list(Sample.dna.col_attrs['id']),
    cytobands=cytobands,
    verbose=True
)

variant_to_amplicon = dict(
    zip(
        np.asarray(Sample.dna.col_attrs["id"]).astype(str),
        np.asarray(Sample.dna.col_attrs["amplicon"]).astype(str),
    )
)

wilcoxon_results_var = CNV_Utils.test_variant_informativeness(
    df_AF=df_AF,
    df_ALT=df_ALT,
    df_REF=df_REF,
    grouped_by_chr_arm=grouped_by_chr_arm,
    output_dir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/allele_frequencies",
    p_threshold=0.01, baf_deviation_threshold=0.3,
    depth_ratio_deletion=0.7, depth_ratio_amplification=1.3,
    require_clone_imbalance=False,
    max_af_iqr=0.35, min_cells=5,
    variant_to_amplicon=variant_to_amplicon,
    germline_variants=list(filtered_germline_variants)+ list(res["new_kept"]),
    verbose=True,
)

### CNV integrated plot

In [ ]:
OUT_DIR = f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/ploidy"
os.makedirs(OUT_DIR, exist_ok=True)

Sample.cnv.normalize_reads()

out = CNV_Utils.extract_cnv_data(
    Sample=Sample,
    desired_plotting_order=desired_plotting_order,
    wd=wd,
    EXPERIMENT=EXPERIMENT,
    PATIENT_ID=PATIENT_ID,
    diploid_label="WT",
    ploidy_before_sync=True,   # matches your current workflow
    output_tag="",             # default filenames
    save_outputs=True,
    verbose=True,
)

# same variables you used before
OUT_DIR = out["OUT_DIR"]
shared_barcodes = out["shared_barcodes"]
variant_amplicons = out["variant_amplicons"]

df_ploidy = out["df_ploidy"]
df_ploidy_median = out["df_ploidy_median"]

df_mut_to_amplicon_dataframe = out["df_mut_to_amplicon_dataframe"]
df_mut_to_amplicon = out["df_mut_to_amplicon"]
variant_to_amp_dict = out["variant_to_amp_dict"]

df_cnv = out["df_cnv"]
df_cnv_grouped = out["df_cnv_grouped"]
df_cnv_vars = out["df_cnv_vars"]
cnv_info = out["cnv_info"]


outputs = {
    "Cell_x_ploidy_data.csv": df_cnv,
    "Clone_x_ploidy_data.csv": df_cnv_grouped,
    "Variants_x_ploidy_data.csv": df_cnv_vars,
    "Ploidy_metadata.csv": cnv_info,
}

for filename, df_out in outputs.items():
    path = f"{OUT_DIR}/{filename}"
    df_out.to_csv(path)
    print(f"Saved: {path}")

In [ ]:
import pandas as pd
import missionbio.mosaic as ms

Sample_raw = ms.load(
    H5_PATH, filter_variants=True, filter_cells=False,
    variant_filter_config={
        "min_dp": 10, "min_gq": 20, "vaf_ref": 5, "vaf_hom": 95, "vaf_het": 30,
        "min_prct_cells": 50,
        "min_mut_prct_cells": max(5, min(20, round(0.01 * sample_raw_size))) * 100 / sample_raw_size,
        "iterations": 10, "n_cores": num_cores,
    },
    whitelist=list(whitelist) + list(germline_whitelist) + list(res["new_kept"]),
)

# match to Sample's cells across all assays
raw_bc = {str(b): b for b in Sample_raw.dna.barcodes()}
keep_bc = [raw_bc[str(b)] for b in Sample.dna.barcodes() if str(b) in raw_bc]
for name in ("dna", "cnv", "protein"):
    a = getattr(Sample_raw, name, None)
    if a is not None:
        setattr(Sample_raw, name, a[keep_bc, a.ids()])

# transfer clone labels from Sample
lbl = pd.Series(Sample.dna.get_labels(), index=[str(b) for b in Sample.dna.barcodes()]).astype(str)
Sample_raw.dna.set_labels(lbl.reindex([str(b) for b in Sample_raw.dna.barcodes()]).to_numpy())

print(f"Sample_raw: {Sample_raw.dna.shape} | {Sample_raw.cnv.shape[1]} amplicons | "
      f"{len(keep_bc)}/{Sample.dna.shape[0]} cells matched")

Sample_all_vars = Sample_raw
Sample_all_vars.cnv.normalize_reads()

out_all = CNV_Utils.extract_cnv_data(
    Sample=Sample_all_vars,
    desired_plotting_order=desired_plotting_order,
    wd=wd, EXPERIMENT=EXPERIMENT, PATIENT_ID=PATIENT_ID,
    diploid_label="WT",
    sample_filter_source=Sample,
    label_source_sample=Sample,
    subset_cnv_to_variant_amplicons=False,
    ploidy_before_sync=False,
    output_tag="_all_vars",
    save_outputs=True,
    verbose=True,
)

del(Sample_raw)

In [ ]:
# Same variables as your old script
OUT_DIR = out_all["OUT_DIR"]
shared_barcodes_all = out_all["shared_barcodes"]
variant_amplicons_all = out_all["variant_amplicons"]

df_ploidy = out_all["df_ploidy"]
df_ploidy_median = out_all["df_ploidy_median"]

df_mut_to_amplicon_all_dataframe = out_all["df_mut_to_amplicon_dataframe"]
df_mut_to_amplicon_all = out_all["df_mut_to_amplicon"]
variant_to_amp_dict = out_all["variant_to_amp_dict"]

df_cnv_all = out_all["df_cnv"]
df_cnv_grouped_all = out_all["df_cnv_grouped"]
df_cnv_vars_all = out_all["df_cnv_vars"]
cnv_info_all = out_all["cnv_info"]

outputs = {
    "Cell_x_ploidy_data_all_vars.csv": df_cnv_all,
    "Clone_x_ploidy_data_all_vars.csv": df_cnv_grouped_all,
    "Variants_x_ploidy_data_all_vars.csv": df_cnv_vars_all,
    "Ploidy_metadata_all_vars.csv": cnv_info_all,
}

for filename, df_out in outputs.items():
    path = f"{OUT_DIR}/{filename}"
    df_out.to_csv(path)
    print(f"Saved: {path}")

# Reshape df_cnv_grouped_all: amplicons as rows, clones as columns → cells as rows, amplicons as columns
# This matches the format expected by df_cnv_all (same shape as df_cnv_vars)

df_cnv_all_reshaped = df_cnv_grouped_all.T.copy()
df_cnv_all_reshaped.index.name = None

# The column names are amplicon IDs — we need to map them to variant-style coords
# Build amplicon_id -> "chrN:pos" mapping from cnv col_attrs
cnv_attrs = Sample_backup.cnv.col_attrs
amp_to_coord = {
    aid: f"chr{chrom}:{end}"
    for aid, chrom, end in zip(cnv_attrs["id"], cnv_attrs["CHROM"], cnv_attrs["end_pos"])
}

df_cnv_all_reshaped.columns = [amp_to_coord.get(c, c) for c in df_cnv_all_reshaped.columns]
df_cnv_all_reshaped["label"] = df_cnv_all_reshaped.index

wt_row = df_cnv_all_reshaped[df_cnv_all_reshaped["label"] == "WT"]
numeric_cols = [c for c in df_cnv_all_reshaped.columns if c != "label"]
nonzero_cols = [c for c in numeric_cols if (wt_row[c].values[0] != 0)]
df_cnv_all_reshaped = df_cnv_all_reshaped[nonzero_cols + ["label"]]

In [ ]:
def get_labeled_dna_attribute(sample, attr):
    """
    Return a row+col DNA attribute matrix with clone/cell labels
    inserted as the first column.
    """
    df = sample.dna.get_attribute(attr, constraint="row+col").copy()
    variant_cols = df.columns.tolist()

    df.insert(0, "label", sample.dna.get_labels())

    return df, variant_cols


# Extract depth and allele frequency matrices
df_dp_all, dp_cols = get_labeled_dna_attribute(Sample_all_vars, "DP")
df_AF_all, af_cols = get_labeled_dna_attribute(Sample_all_vars, "AF")

# Build numeric AF/DP matrices
DP = df_dp_all.drop(columns="label").fillna(0)
AF = df_AF_all.drop(columns="label").fillna(0)

# Ensure AF and DP have identical columns/order
AF = AF.reindex(columns=DP.columns, fill_value=0)

# Compute approximate ALT and REF read counts
df_ALT_all = (AF / 100) * DP
df_REF_all = DP - df_ALT_all

In [ ]:
CNV_Utils.plot_integrated_cnv_baf_with_significance(
    df_AF=df_AF,
    df_cnv_vars=df_cnv_all_reshaped,
    df_ALT=df_ALT,
    df_REF=df_REF,
    cytobands=cytobands,
    wilcoxon_results_var=wilcoxon_results_var,
    df_mut_to_amplicon=df_mut_to_amplicon,
    differential_variants=differential_variants,
    desired_plotting_order=desired_plotting_order,
    output_path=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clones_CNV_BAF_Significant.png",
    height_baf=1.75, height_cnv=1.75, figure_width=20,
    height_alt=1, height_ref=1,
    hspace=0.3,
    height_multiplier=0.85, height_spacer=0.5,
    max_jitter_cells=100,
    cytoband_fontsize=12,
    font_chr_arms=14,
    font_yticks=12,
    font_ylabel=12,
    font_genotype=12,
    plot_separate_clones=True,
    color_cytoband_background_in_ploidy=False,
    plot_ploidy_median=True,
    plot_title=False,
    dpi=300,
    verbose=True
)

In [ ]:
# Count variants that fall on the selected arms
chr_arms_to_plot = ['chr17p']

all_germline = [c for c in df_AF.columns if isinstance(c, str) and c.startswith("chr")]
n_variants = sum(
    1 for v in all_germline
    if any(v.startswith(arm.replace("chr", "chr").split("p")[0].split("q")[0]) for arm in chr_arms_to_plot)
)
# More robust: just count after filtering — but a simple heuristic works
# ~0.3-0.5 inches per variant is a good starting point
figure_width = max(0.4, n_variants * 0.1)

CNV_Utils.plot_integrated_cnv_baf_with_significance(
    df_AF=df_AF,
    df_cnv_vars=df_cnv_all_reshaped,
    df_ALT=df_ALT,
    df_REF=df_REF,
    cytobands=cytobands,
    wilcoxon_results_var=wilcoxon_results_var,
    df_mut_to_amplicon=df_mut_to_amplicon,
    differential_variants=differential_variants,
    desired_plotting_order=desired_plotting_order,
    output_path=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clones_CNV_BAF_Significant_chr17p.png",
    height_baf=1.25, height_cnv=1,
    height_alt=0.5, height_ref=0.5,
    figure_width=figure_width,
    height_multiplier=0.85, height_spacer=0.5,
    hspace=0.3,
    chr_arms_to_plot=chr_arms_to_plot,
    max_jitter_cells=100,
    font_yticks=10,
    font_ylabel=10,
    font_genotype=10,
    font_chr_arms=10,
    cytoband_fontsize=10,
    plot_separate_clones=True,
    plot_ploidy_median=True,
    plot_alt_ref=False,
    rotation_chr_labels=0,
    show_ylabel=False,
    plot_title=False,
    dpi=300,
    show_chr_arm_labels=False,
    verbose=True,
)

In [ ]:
# Count variants that fall on the selected arms
chr_arms_to_plot = ['chr5q', 'chr7q', 'chr17p']

all_germline = [c for c in df_AF.columns if isinstance(c, str) and c.startswith("chr")]
n_variants = sum(
    1 for v in all_germline
    if any(v.startswith(arm.replace("chr", "chr").split("p")[0].split("q")[0]) for arm in chr_arms_to_plot)
)
# More robust: just count after filtering — but a simple heuristic works
# ~0.3-0.5 inches per variant is a good starting point
figure_width = max(1, n_variants * 0.1)

CNV_Utils.plot_integrated_cnv_baf_with_significance(
    df_AF=df_AF,
    df_cnv_vars=df_cnv_all_reshaped,
    df_ALT=df_ALT,
    df_REF=df_REF,
    cytobands=cytobands,
    wilcoxon_results_var=wilcoxon_results_var,
    df_mut_to_amplicon=df_mut_to_amplicon,
    differential_variants=differential_variants,
    desired_plotting_order=desired_plotting_order,
    output_path=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clones_CNV_BAF_Significant_selected.png",
    height_baf=1.25, height_cnv=1,
    height_alt=0.5, height_ref=0.5,
    figure_width=figure_width,
    height_multiplier=0.85, height_spacer=0.5,
    hspace=0.3,
    chr_arms_to_plot=chr_arms_to_plot,
    max_jitter_cells=100,
    font_yticks=10,
    font_ylabel=10,
    font_genotype=10,
    font_chr_arms=10,
    cytoband_fontsize=10,
    plot_separate_clones=True,
    plot_ploidy_median=True,
    plot_alt_ref=False,
    rotation_chr_labels=90,
    show_ylabel=False,
    plot_title=False,
    dpi=300,
    show_chr_arm_labels=True,
    verbose=True,
)

In [ ]:
# # Rename clones if required
rename_map = {
    "":"",
}

# Sample.dna.rename_labels(rename_map)

# desired_plotting_order = [
#     ""
# ]

In [ ]:
from pathlib import Path

def build_loh_table(sample, loh_rules, default_state="WT", af_attr="AF"):
    df_af = sample.dna.get_attribute(af_attr, constraint="row+col").copy()
    variant_cols = df_af.columns.tolist()
    df_af["label"] = sample.dna.get_labels()
    df_af = df_af[["label"] + variant_cols]

    df_loh = pd.DataFrame(index=df_af.index)
    df_loh["label"] = df_af["label"]

    for arm, mapping in loh_rules.items():
        df_loh[arm] = default_state
        for clone_label, state in mapping.items():
            df_loh.loc[df_loh["label"] == clone_label, arm] = state

    return df_loh


loh_rules = {
    "chr5q": {"TP53_R249T_Hem": "Del"},
    "chr7q": {"TP53_R249T_Hem": "Del"},
    "chr17p": {"TP53_R249T_Hem": "Del"},
}

df_LOH = build_loh_table(Sample, loh_rules)

folder_to_save = Path(wd) / "results" / "per_sample" / EXPERIMENT / PATIENT_ID / "dna_cnv" / "matrices"
folder_to_save.mkdir(parents=True, exist_ok=True)

out_path = folder_to_save / "Copy_Number_Alterations.csv"
df_LOH.to_csv(out_path, index=True)
print(f"Saved: {out_path}")

## VAFs and clone sizes

In [ ]:
df_vaf = DNA_Utils.get_VAF(
    Sample,
    label_col='label',
    clone_order=desired_plotting_order,
    output_path=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Pseudobulk_VAFs.csv",
)

# Display
df_vaf

In [ ]:
clone_counts = DNA_Utils.get_clones_size(
    Sample,
    label_col='label',
    desired_order=desired_plotting_order
)

# Save to CSV
clone_counts.to_csv(
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Clone_absolute_counts.csv",
    index=False
)

plt.figure(figsize=(4, 3))
barplot = sns.barplot(y='Clone', x='Count', data=clone_counts)

for index, row in clone_counts.iterrows():
    barplot.text(row['Count'], index, str(int(row['Count'])),
                color='black', ha="center", va="center", fontweight='bold')

plt.title('Cell Counts per Clone')
plt.xlabel('Cell Count')
plt.tight_layout()
plt.savefig(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clones_absolute_number.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
clone_relative_counts = DNA_Utils.get_clones_size(
    Sample,
    label_col='label',
    desired_order=desired_plotting_order,
    return_percentage=True
)

# Save to CSV
clone_relative_counts.to_csv(
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Clone_relative_counts.csv",
    index=False
)

plt.figure(figsize=(4, 3))
barplot = sns.barplot(y='Clone', x='Percentage', data=clone_relative_counts)

for index, row in clone_relative_counts.iterrows():
    barplot.text(row['Percentage'], index, str(round(row['Percentage'],1)),
                color='black', ha="center", va="center", fontweight='bold')

plt.title('Relative percentage per Clone')
plt.xlabel('Relative percentage')
plt.tight_layout()
plt.savefig(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clones_relative_number.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
clone_counts = DNA_Utils.get_clones_size(
    Sample,
    label_col='label',
    desired_order=desired_plotting_order,
    exclude_celltypes=['CD8_T', 'CD4_T', 'Other_T', 'B', 'Plasma']
)

# Save to CSV
clone_counts.to_csv(
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Clone_absolute_counts_no_lymphoid.csv",
    index=False
)

plt.figure(figsize=(4, 3))
barplot = sns.barplot(y='Clone', x='Count', data=clone_counts)

for index, row in clone_counts.iterrows():
    barplot.text(row['Count'], index, str(int(row['Count'])),
                color='black', ha="center", va="center", fontweight='bold')

plt.title('Cell Counts per Clone')
plt.xlabel('Cell Count')
plt.tight_layout()
plt.savefig(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clone_absolute_counts_no_lymphoid.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
clone_relative_counts = DNA_Utils.get_clones_size(
    Sample,
    label_col='label',
    desired_order=desired_plotting_order,
    return_percentage=True,
    exclude_celltypes=['CD8_T', 'CD4_T', 'Other_T', 'B', 'Plasma']
)

# Save to CSV
clone_relative_counts.to_csv(
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Clone_relative_counts_no_lymphoid.csv",
    index=False
)

plt.figure(figsize=(4, 3))
barplot = sns.barplot(y='Clone', x='Percentage', data=clone_relative_counts)

for index, row in clone_relative_counts.iterrows():
    barplot.text(row['Percentage'], index, str(round(row['Percentage'],1)),
                color='black', ha="center", va="center", fontweight='bold')

plt.title('Relative percentage per Clone')
plt.xlabel('Relative percentage')
plt.tight_layout()
plt.savefig(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes/Clone_relative_counts_no_lymphoid.png", dpi=300, bbox_inches='tight')
plt.show()

## Final plots

In [ ]:
fig = PRO_Utils.plot_protein_umap(
    Sample=Sample,
    celltype_column='label',
    outdir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/protein/annotation",
    output_filename="Protein_UMAP_celltypes.png",
    point_size=8,
    width=600,
    height=600,
    show_plot=True,
    verbose=True
)

In [ ]:
# Plot all differential variants
figures = DNA_Utils.plot_somatic_variants(
    Sample=Sample,
    differential_variants=differential_variants,
    variant_metadata=merged_annotation_filtered_detected_variants,
    outdir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes",
    output_prefix="Variant_UMAP",
    variant_color='#191919',  # Red-orange
    background_color='#D3D3D3',  # Light grey
    variant_size=6,
    background_size=3,
    width=200,
    height=200,
    show_plots=True,
    verbose=True
)

In [ ]:
figures = DNA_Utils.plot_individual_clone_umaps(
    Sample=Sample,
    clone_column='label',
    outdir=f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/clonotypes",
    output_prefix="Clone_UMAP",
    umap_attribute="umap",
    background_color='#adadad',  # Light grey
    clone_size=6,
    background_size=3,
    width=200,
    height=200,
    verbose=True
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def resize_keep_aspect(img, target_height=None, target_width=None):
    w, h = img.size

    if target_height is not None:
        new_w = int(w * target_height / h)
        new_h = target_height
    elif target_width is not None:
        new_w = target_width
        new_h = int(h * target_width / w)
    else:
        return img

    return img.resize((new_w, new_h), Image.LANCZOS)


def show_side_by_side(
    umap_path,
    cnv_path,
    clone_name,
    umap_height=1400,
    cnv_height=1400,
    out_path=None
):
    umap = Image.open(umap_path)
    cnv = Image.open(cnv_path)

    # Resize each image while keeping aspect ratio
    umap = resize_keep_aspect(umap, target_height=umap_height)
    cnv = resize_keep_aspect(cnv, target_height=cnv_height)

    # Get the resized dimensions
    w1, h1 = umap.size
    w2, h2 = cnv.size

    # Make the canvas match the image heights
    dpi = 100
    fig_h = max(h1, h2) / dpi
    fig_w = (w1 + w2) / dpi

    fig, axes = plt.subplots(
        1, 2,
        figsize=(fig_w, fig_h),
        gridspec_kw={"width_ratios": [w1, w2]}
    )

    axes[0].imshow(umap)
    axes[0].axis("off")
    #axes[0].set_title(f"{clone_name} - UMAP")

    axes[1].imshow(cnv)
    axes[1].axis("off")
    #axes[1].set_title(f"{clone_name} - CNV")

    fig.suptitle(clone_name, fontsize=40, fontweight="bold")
    plt.subplots_adjust(wspace=0.02, top=0.90)

    if out_path is not None:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
import os
import re
import glob
import pandas as pd

CLONOTYPE_DIR = (
    f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}"
    "/dna_cnv/clonotypes"
)

reverse_rename_map = {v: k for k, v in rename_map.items()}

def _safe_label_for_filename(label):
    return (
        str(label)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )


def _dedupe(xs):
    out = []
    seen = set()

    for x in xs:
        x = str(x).strip()
        if not x or x in seen:
            continue

        out.append(x)
        seen.add(x)

    return out


def _display_label(label):
    """
    Convert old internal/raw clone label to canonical display label.
    """
    label = str(label).strip()
    return rename_map.get(label, label)


def _label_aliases(label):
    """
    Return all possible names that may have been used when saving files.

    Handles:
      old → new
      new → old
      LOH_Chr ↔ LOH_chr variants
      LOH_Chr4q → LOH-C0 chr4q / LOH-C1 chr4q variants
    """
    label = str(label).strip()

    aliases = [label]

    # old -> new
    if label in rename_map:
        aliases.append(rename_map[label])

    # new -> old
    if label in reverse_rename_map:
        aliases.append(reverse_rename_map[label])

    expanded = []

    for x in aliases:
        expanded.extend(
            [
                x,
                x.replace("_Chr", "_chr"),
                x.replace(" LOH_Chr", " LOH_"),
                x.replace(" LOH_Chr", " LOH-C0 chr"),
                x.replace(" LOH_Chr", " LOH-C1 chr"),
            ]
        )

    return _dedupe(expanded)


def _find_clone_image(clone_name, prefix, directory=CLONOTYPE_DIR):
    """
    Search for image file using current label plus rename aliases.
    """
    candidates = []

    for alias in _label_aliases(clone_name):
        safe_alias = _safe_label_for_filename(alias)

        candidates.extend(
            [
                os.path.join(directory, f"{prefix}_{safe_alias}.png"),
                os.path.join(directory, f"{prefix}_{safe_alias}.pdf"),
            ]
        )

    for path in candidates:
        if os.path.exists(path):
            return path

    # Last-resort fuzzy search across all aliases.
    for alias in _label_aliases(clone_name):
        safe_alias = _safe_label_for_filename(alias)

        fuzzy_patterns = [
            os.path.join(directory, f"{prefix}*{safe_alias}*.png"),
            os.path.join(directory, f"{prefix}*{safe_alias}*.pdf"),
        ]

        for pattern in fuzzy_patterns:
            hits = sorted(glob.glob(pattern))
            if hits:
                return hits[0]

    # Return first candidate for informative missing-file printout.
    return candidates[0] if candidates else None


clone_series_raw = (
    pd.Series(Sample.dna.row_attrs["Clone"])
    .astype(str)
    .str.strip()
)

clone_series_display = clone_series_raw.map(_display_label)

clone_counts = clone_series_display.value_counts(dropna=False)

# Map display label -> raw labels currently present in Sample.
display_to_raw = {}

for raw_label, display_label in zip(clone_series_raw, clone_series_display):
    display_to_raw.setdefault(display_label, [])
    if raw_label not in display_to_raw[display_label]:
        display_to_raw[display_label].append(raw_label)


if "desired_plotting_order" in globals() and desired_plotting_order is not None:
    clone_order = [
        str(c).strip()
        for c in desired_plotting_order
        if str(c).strip() in set(clone_series_display)
    ]

    clone_order += [
        c for c in clone_counts.index.astype(str).tolist()
        if c not in clone_order
    ]
else:
    def _natural_key(x):
        return [
            int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(x))
        ]

    clone_order = sorted(
        clone_counts.index.astype(str).tolist(),
        key=_natural_key,
    )


clones = {}

for display_clone_name in clone_order:
    raw_aliases = display_to_raw.get(display_clone_name, [])
    search_labels = _dedupe([display_clone_name] + raw_aliases + _label_aliases(display_clone_name))

    cnv_baf_path = None
    umap_path = None

    for label in search_labels:
        candidate = _find_clone_image(
            label,
            prefix="Clones_CNV_BAF_Significant_selected",
        )
        if candidate and os.path.exists(candidate):
            cnv_baf_path = candidate
            break

    if cnv_baf_path is None:
        cnv_baf_path = _find_clone_image(
            display_clone_name,
            prefix="Clones_CNV_BAF_Significant_selected",
        )

    for label in search_labels:
        candidate = _find_clone_image(
            label,
            prefix="Clone_UMAP",
        )
        if candidate and os.path.exists(candidate):
            umap_path = candidate
            break

    if umap_path is None:
        umap_path = _find_clone_image(
            display_clone_name,
            prefix="Clone_UMAP",
        )

    clones[display_clone_name] = {
        "raw_aliases": raw_aliases,
        "cnv_baf": cnv_baf_path,
        "umap": umap_path,
    }


print("Clone image pairs:")

for clone_name, paths in clones.items():
    print(f"\n{clone_name}")

    if paths["raw_aliases"]:
        print(f"  raw aliases: {paths['raw_aliases']}")

    print(f"  CNV/BAF exists: {os.path.exists(paths['cnv_baf'])} | {paths['cnv_baf']}")
    print(f"  UMAP    exists: {os.path.exists(paths['umap'])} | {paths['umap']}")


for clone_name, paths in clones.items():
    if not os.path.exists(paths["cnv_baf"]):
        print(f"Skipping {clone_name}: missing CNV/BAF {paths['cnv_baf']}")
        continue

    if not os.path.exists(paths["umap"]):
        print(f"Skipping {clone_name}: missing UMAP {paths['umap']}")
        continue

    show_side_by_side(
        paths["cnv_baf"],
        paths["umap"],
        clone_name=clone_name,
        # out_path=f"{_safe_label_for_filename(clone_name)}.png"
    )

## Export

In [ ]:
pro_df_anno.shape

In [ ]:
folder_to_save = f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/protein/matrices"

if not os.path.exists(folder_to_save):
    os.makedirs(folder_to_save)

for assay in [Sample.protein]:
    if assay is not None:
        if not os.path.exists(f"{folder_to_save}/{assay.name}"):
            os.makedirs(f"{folder_to_save}/{assay.name}")

        if not os.path.exists(f"{folder_to_save}/{assay.name}/Filtered"):
            os.makedirs(f"{folder_to_save}/{assay.name}/Filtered")

        for layer in assay.layers.keys():
            df = assay.get_attribute(layer, constraint="row+col")
            cols = list(df.columns.values)

            df.loc[:, "label"] = assay.get_labels()
            df = df.loc[:, ["label"] + cols]

            df.to_csv(f"{folder_to_save}/{assay.name}/Filtered/{layer}.csv")

pro_df_anno.to_csv(
    f"{folder_to_save}/{Sample.protein.name}/Filtered/read_counts_all_cells.csv"
)

pro_df_anno_raw.to_csv(
    f"{folder_to_save}/{Sample.protein.name}/Filtered/raw_read_counts_all_cells.csv"
)

In [ ]:
folder_to_save = f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices"
      
if not os.path.exists(folder_to_save):
    os.makedirs(folder_to_save)    

for assay in [Sample.dna]:
    if assay is not None:
        if not os.path.exists(f'{folder_to_save}/{assay.name}'):
            os.makedirs(f'{folder_to_save}/{assay.name}')
        if not os.path.exists(f'{folder_to_save}/{assay.name}/Filtered'):
            os.makedirs(f'{folder_to_save}/{assay.name}/Filtered')

        for layer in assay.layers.keys():
            df = assay.get_attribute(layer, constraint='row+col')
            cols = list(df.columns.values)
            df.loc[:, 'label'] = assay.get_labels()
            df = df.loc[:, ['label'] + cols]
            df.to_csv(f'{folder_to_save}/{assay.name}/Filtered/{layer}.csv')

In [ ]:
number_of_variants = len(Sample.dna.col_attrs['id'])

subsets = []
Annotation_filtered_detected_variants = []

split_samples = [Sample.dna[:, i:i+1000] for i in range(0, number_of_variants, 1000)]

counter = 0

for i, subset in enumerate(split_samples):
    subsets.append(copy.copy(Sample))
    subsets[counter].dna = subset
    Annotation_filtered_detected_variants.append(subsets[counter].dna.get_annotations())
    counter += 1
    # Save or process new_sample as needed

# Assuming Annotation_filtered_detected_variants is a list of data frames
merged_annotation_filtered_detected_variants = pd.concat(Annotation_filtered_detected_variants, axis=0)

merged_annotation_filtered_detected_variants.to_csv(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/matrices/Significant_captured_variants_annotation.csv")

In [ ]:
df = Sample.dna.get_attribute('AF', constraint='row+col')
cols = list(df.columns.values)
df.loc[:, 'label'] = Sample.dna.get_labels()
df = df.loc[:, ['label'] + cols]

if not variable_af_germline:
    if verbose:
        print("No variable germline variants found (variable_af_germline is empty). Skipping export.")
else:
    df_variable = df.loc[:, ['label'] + variable_af_germline]
    df_variable.to_csv(
        f"{folder_to_save}/AF_variable_germline_het_mutations.csv",
        sep=",",
        header=True,
        index=True
    )

df_non_variable = df.loc[:, ['label'] + non_variable_af_germline]
df_non_variable.to_csv(f'{folder_to_save}/AF_non_variable_germline_het_mutations.csv', sep=',', header=True)

df_somatic = df.loc[:, ['label'] + differential_variants]
df_somatic.to_csv(f'{folder_to_save}/AF_somatic_mutations.csv', sep=',', header=True)

In [ ]:
df = Sample.dna.get_attribute('NGT', constraint='row+col')
cols = list(df.columns.values)
df.loc[:, 'label'] = Sample.dna.get_labels()
df = df.loc[:, ['label'] + cols]

if not variable_af_germline:
    if verbose:
        print("No variable germline variants found (variable_af_germline is empty). Skipping export.")
else:
    df_variable = df.loc[:, ['label'] + variable_af_germline]
    df_variable.to_csv(
        f"{folder_to_save}/NGT_variable_germline_het_mutations.csv",
        sep=",",
        header=True,
        index=True
    )

df_non_variable = df.loc[:, ['label'] + non_variable_af_germline]
df_non_variable.to_csv(f'{folder_to_save}/NGT_non_variable_germline_het_mutations.csv', sep=',', header=True)

df_somatic = df.loc[:, ['label'] + differential_variants]
df_somatic.to_csv(f'{folder_to_save}/NGT_somatic_mutations.csv', sep=',', header=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPORT: full (unfiltered) variant space × selected cell set
# ══════════════════════════════════════════════════════════════════════════════

# ── Settings ──────────────────────────────────────────────────────────────────
VARIANT_FILTER_CONFIG = {
    "min_dp": 10,
    "min_gq": 20,
    "vaf_ref": 5,
    "vaf_hom": 95,
    "vaf_het": 30,
    "min_prct_cells": 10,
    "min_mut_prct_cells": 0,
    "n_cores": num_cores,
}

# Which cells to export.
#   "all"          – every barcode on the DNA assay (no cell-level subsetting)
#   "protein"      – barcodes present in PROTEIN_TABLE (protein-passing cells)
#   "aberrancies"  – df_aberrancies if populated, else df_somatic
CELL_SET      = "protein"
PROTEIN_TABLE = "pro_df_anno"   # name of the already-built protein frame in the kernel

EXPORT_FORMATS    = ("csv",)  # NOTE trailing comma — ("csv") is the *string* "csv",
                              # which tuple()s to ('c','s','v') and writes .c/.s/.v files
ANNOTATE          = True
ANNOT_CHUNK_SIZE  = 1000
USE_CACHE_LOCK    = True      # serialise access to the shared Varsome cache
REPAIR_CACHE      = True      # repair-and-retry a chunk that hits a ParserError
CHECK_CACHE_FIRST = True      # inspect + repair before the run rather than mid-loop

folder_to_save = f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/dna_cnv/variants"
out_dir        = f"{folder_to_save}/all_variants"

# ── Varsome cache pre-flight ──────────────────────────────────────────────────
# The cache lives inside site-packages and is shared by every process in the
# env, so a previous concurrent run can have left torn lines in it. Repairing
# up front is cheaper than discovering it 40 chunks into the annotation loop.
if ANNOTATE and CHECK_CACHE_FIRST:
    print("=" * 70)
    print("VARSOME CACHE PRE-FLIGHT")
    print("=" * 70)
    cache_status = DNA_Utils.inspect_varsome_db()
    if cache_status["n_bad"] or cache_status["n_duplicate_keys"]:
        print("  → repairing")
        DNA_Utils.repair_varsome_db()
    else:
        print("  ✓ Cache clean")

# ── Load ──────────────────────────────────────────────────────────────────────
Sample_unfiltered = ms.load(
    H5_PATH,
    filter_variants=True,
    filter_cells=False,
    variant_filter_config=VARIANT_FILTER_CONFIG,
)
print(f"Loaded: {Sample_unfiltered.dna.shape[0]} cells "
      f"x {Sample_unfiltered.dna.shape[1]} variants")

# ── Choose the cell set ───────────────────────────────────────────────────────
assay_barcodes = np.asarray(Sample_unfiltered.dna.barcodes(), dtype=str)

if CELL_SET == "all":
    # Every barcode on the DNA assay. subset_assay_to_cells accepts a plain array
    # as readily as a DataFrame index, and preserve_assay_order makes this a
    # no-op reordering rather than a real subset.
    cell_table  = assay_barcodes
    cell_source = "all assay barcodes"
    _idx_sample = [str(b) for b in assay_barcodes[:5]]

elif CELL_SET == "protein":
    # Cells with protein data, from the frame already built in the kernel. Its
    # 'label' and AbO columns are ignored — only the barcode index is used.
    # Note the protein frame is typically derived from `Sample`, which may be
    # cell-filtered, whereas the export runs on Sample_unfiltered; the overlap
    # check below makes any difference explicit.
    _prot = globals().get(PROTEIN_TABLE)
    if _prot is None or not hasattr(_prot, "empty") or _prot.empty:
        raise AssertionError(
            f"'{PROTEIN_TABLE}' is missing or empty in the kernel — build it "
            f"before exporting with CELL_SET='protein', or set PROTEIN_TABLE "
            f"to the correct variable name."
        )
    cell_table  = _prot
    cell_source = PROTEIN_TABLE
    _idx_sample = [str(i) for i in _prot.index[:5]]
    print(f"Protein table '{PROTEIN_TABLE}': {_prot.shape[0]} cells × "
          f"{_prot.shape[1]} columns")

elif CELL_SET == "aberrancies":
    # Prefer the aberrancy table (somatic + LOH); fall back to somatic-only.
    if "df_aberrancies" in dir() and df_aberrancies is not None and not df_aberrancies.empty:
        cell_table  = df_aberrancies
        cell_source = "df_aberrancies"
    else:
        cell_table  = df_somatic
        cell_source = "df_somatic"
        print("⚠️  Using df_somatic index — no LOH cells to merge in")

    if cell_table is None or cell_table.empty:
        raise AssertionError(
            f"Cell table '{cell_source}' is empty — nothing to subset to."
        )
    _idx_sample = [str(i) for i in cell_table.index[:5]]

else:
    raise AssertionError(
        f"CELL_SET must be 'all', 'protein' or 'aberrancies', got {CELL_SET!r}"
    )

# Guard against passing a variant-indexed table by mistake: cell tables are
# indexed by barcode, and subset_assay_to_cells would otherwise raise deep in
# the call stack rather than here.
if CELL_SET != "all" and any(":" in i and "/" in i for i in _idx_sample):
    raise AssertionError(
        f"'{cell_source}' looks variant-indexed (e.g. {_idx_sample[:2]}), "
        f"not cell-indexed. Pass a per-cell table."
    )

n_requested = len(cell_table) if CELL_SET == "all" else cell_table.shape[0]
print(f"\nCell set: {n_requested} barcodes from {cell_source}, e.g. {_idx_sample[:3]}")

# ── Cell-set overlap diagnostics ──────────────────────────────────────────────
# Report the relationship between the requested set and the DNA assay before
# committing to a long annotation run, so a surprising cell count surfaces now.
if CELL_SET == "all":
    for _name in (PROTEIN_TABLE, "df_aberrancies", "df_somatic"):
        _tbl = globals().get(_name)
        if _tbl is not None and hasattr(_tbl, "empty") and not _tbl.empty:
            _n_excl = len(set(assay_barcodes) - set(map(str, _tbl.index)))
            print(f"  ({_n_excl} cell(s) absent from {_name} — included anyway)")
else:
    _req   = set(map(str, cell_table.index))
    _assay = set(assay_barcodes)
    print(f"  In DNA assay and {cell_source}: {len(_assay & _req)}")
    print(f"  In DNA assay only (excluded):  {len(_assay - _req)}")
    print(f"  In {cell_source} only (no DNA): {len(_req - _assay)}")
    if not (_assay & _req):
        raise AssertionError(
            f"No barcode overlap between the DNA assay and {cell_source}. "
            f"Check whether the two objects use the same barcode formatting "
            f"(e.g. a '-1' suffix or a '__PTxx' sample tag on one side only)."
        )

# ── Export ────────────────────────────────────────────────────────────────────
res = DNA_Utils.export_cell_subset_matrices(
    Sample_unfiltered.dna,
    cell_table,
    out_dir,
    annotate=ANNOTATE,
    annot_chunk_size=ANNOT_CHUNK_SIZE,
    use_cache_lock=USE_CACHE_LOCK,
    repair_on_parse_error=REPAIR_CACHE,
    formats=EXPORT_FORMATS,
)

# ── Post-run checks ───────────────────────────────────────────────────────────
n_exported = len(res["assay"].barcodes())
if CELL_SET == "all" and n_exported != len(assay_barcodes):
    raise AssertionError(
        f"Requested all {len(assay_barcodes)} assay barcodes but exported "
        f"{n_exported}. Check for duplicate barcodes on the assay."
    )
print(f"\nExported {n_exported} cells")

if res["annot_failed"]:
    print(f"\n⚠️  {len(res['annot_failed'])} annotation chunk(s) failed — "
          f"those variants are NaN in the metadata table:")
    for start, stop, err in res["annot_failed"]:
        print(f"    [{start}:{stop}]  {err[:120]}")

if res["metadata"] is not None:
    n_annot = int(res["metadata"]["Gene"].notna().sum()) if "Gene" in res["metadata"] else 0
    print(f"\nMetadata: {res['metadata'].shape[0]} variants × "
          f"{res['metadata'].shape[1]} fields ({n_annot} with a Gene call)")

print("\nWritten:")
for key, val in res["paths"].items():
    print(f"  {key}: {val}")

In [ ]:
if df_LOH is None or df_LOH.empty:
    print("No variable germline variants found (df_LOH is empty). Skipping export.")
else:
    df_aberrancies = df_somatic.merge(
        df_LOH,
        left_index=True,
        right_index=True,
        how="left",
        suffixes=("_x", "_y")
    )

    # Drop label_y and rename label_x → label
    if "label_y" in df_aberrancies.columns:
        df_aberrancies = df_aberrancies.drop(columns=["label_y"])

    if "label_x" in df_aberrancies.columns:
        df_aberrancies = df_aberrancies.rename(columns={"label_x": "label"})

    print(df_aberrancies)

    df_aberrancies.to_csv(
        f"{folder_to_save}/All_aberrancies.csv",
        sep=",",
        header=True,
        index=True
    )

## Multi-omic view

In [ ]:
Sample.multi = copy.copy(Sample)
Sample.multi = Sample.multi[Sample.common_barcodes()]

df = Sample.multi.dna.crosstab(
    Sample.multi.protein.row_attrs['Final_annotation'],
    normalize='index',
)
df = df.T.reindex(columns=desired_plotting_order).T

plt.figure(figsize=(8, 3))
sns.set(font_scale=0.1)
ax = sns.heatmap(df)
ax.tick_params(axis='x', labelsize=6, labelrotation=90)
ax.tick_params(axis='y', labelsize=6, labelrotation=0)
for label in ax.get_yticklabels():
    label.set_weight('bold')
plt.savefig(f"{wd}/results/per_sample/{EXPERIMENT}/{PATIENT_ID}/protein/annotation/Clones_Celltype_percentage.png", dpi=300, bbox_inches='tight')
plt.show()